# Dataset Analysis for Capsule Vision Inspection

This notebook analyzes the capsule dataset before training.

## Checks Performed:
- Total images
- Images per class
- Annotations per class
- Missing labels
- Empty labels
- Invalid coordinates
- Duplicate images
- Image dimensions
- Class imbalance

## Charts:
1. Class distribution
2. Bounding-box size distribution
3. Bounding-box center distribution
4. Visual samples with annotations

In [ ]:
# Install dependencies if needed
# !pip install ultralytics matplotlib seaborn

import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Configuration
DATASET_PATH = Path("/content/dataset")  # Change to your dataset path

CLASS_NAMES = [
    "Good",
    "Crack",
    "Scratch",
    "Faulty Imprint",
    "Poke",
    "Squeeze",
    "Contamination",
]

print(f"Dataset path: {DATASET_PATH}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
def load_dataset_stats(dataset_path):
    """Load and analyze the dataset structure."""
    stats = {
        "total_images": 0,
        "images_per_class": Counter(),
        "annotations_per_class": Counter(),
        "missing_labels": [],
        "empty_labels": [],
        "invalid_coords": [],
        "duplicate_images": [],
        "image_dims": [],
        "bbox_sizes": [],
        "bbox_centers": [],
        "splits": {},
    }

    for split in ["train", "val", "test"]:
        img_dir = dataset_path / "images" / split
        lbl_dir = dataset_path / "labels" / split

        if not img_dir.exists():
            print(f"Warning: {img_dir} does not exist")
            continue

        images = sorted(img_dir.glob("*.*"))
        stats["splits"][split] = len(images)
        stats["total_images"] += len(images)

        for img_path in images:
            # Check label file
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if not lbl_path.exists():
                stats["missing_labels"].append(str(img_path))
                continue

            # Read label
            with open(lbl_path) as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]

            if not lines:
                stats["empty_labels"].append(str(img_path))
                continue

            # Parse annotations
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    stats["invalid_coords"].append((str(img_path), line))
                    continue

                try:
                    cls_id = int(parts[0])
                    x_c, y_c, w, h = [float(p) for p in parts[1:]]
                except ValueError:
                    stats["invalid_coords"].append((str(img_path), line))
                    continue

                # Validate coordinates
                if not (0 <= x_c <= 1 and 0 <= y_c <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                    stats["invalid_coords"].append((str(img_path), line))
                    continue

                if cls_id >= len(CLASS_NAMES):
                    stats["invalid_coords"].append((str(img_path), f"Invalid class: {cls_id}"))
                    continue

                stats["annotations_per_class"][cls_id] += 1
                stats["bbox_sizes"].append((w, h))
                stats["bbox_centers"].append((x_c, y_c))

            # Image dimensions
            try:
                with Image.open(img_path) as img:
                    stats["image_dims"].append(img.size)
            except Exception:
                pass

    return stats


stats = load_dataset_stats(DATASET_PATH)

In [ ]:
# Print summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total images: {stats['total_images']}")
print(f"Splits: {stats['splits']}")
print()
print(f"Missing labels: {len(stats['missing_labels'])}")
print(f"Empty labels: {len(stats['empty_labels'])}")
print(f"Invalid coordinates: {len(stats['invalid_coords'])}")
print()
print("Annotations per class:")
for cls_id in range(len(CLASS_NAMES)):
    count = stats["annotations_per_class"].get(cls_id, 0)
    print(f"  {CLASS_NAMES[cls_id]}: {count}")

In [ ]:
# Chart 1: Class distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Images per class (from annotations)
class_counts = [stats["annotations_per_class"].get(i, 0) for i in range(len(CLASS_NAMES))]
ax[0].bar(CLASS_NAMES, class_counts, color="steelblue")
ax[0].set_title("Annotations per Class")
ax[0].set_xlabel("Class")
ax[0].set_ylabel("Count")
ax[0].tick_params(axis="x", rotation=45)

# Split distribution
splits = list(stats["splits"].keys())
split_counts = [stats["splits"][s] for s in splits]
ax[1].bar(splits, split_counts, color="coral")
ax[1].set_title("Images per Split")
ax[1].set_xlabel("Split")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Bounding-box size distribution
if stats["bbox_sizes"]:
    widths = [w for w, h in stats["bbox_sizes"]]
    heights = [h for w, h in stats["bbox_sizes"]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(widths, bins=30, color="steelblue", alpha=0.7)
    axes[0].set_title("Bounding Box Width Distribution")
    axes[0].set_xlabel("Normalized Width")
    axes[0].set_ylabel("Count")

    axes[1].hist(heights, bins=30, color="coral", alpha=0.7)
    axes[1].set_title("Bounding Box Height Distribution")
    axes[1].set_xlabel("Normalized Height")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()

In [ ]:
# Chart 3: Bounding-box center distribution
if stats["bbox_centers"]:
    centers = np.array(stats["bbox_centers"])
    plt.figure(figsize=(8, 8))
    plt.scatter(centers[:, 0], centers[:, 1], alpha=0.3, s=10)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().invert_yaxis()
    plt.title("Bounding Box Center Distribution")
    plt.xlabel("X Center (normalized)")
    plt.ylabel("Y Center (normalized)")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Visual samples with annotations
import random
import cv2

def draw_annotations(img_path, lbl_path, class_names):
    """Draw YOLO annotations on an image."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    with open(lbl_path) as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        x_c, y_c, bw, bh = [float(p) for p in parts[1:]]

        x1 = int((x_c - bw / 2) * w)
        y1 = int((y_c - bh / 2) * h)
        x2 = int((x_c + bw / 2) * w)
        y2 = int((y_c + bh / 2) * h)

        color = (255, 0, 0) if cls_id == 0 else (255, 0, 0)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = class_names[cls_id] if cls_id < len(class_names) else f"Class {cls_id}"
        cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    return img

# Sample 20-30 images
sample_images = []
for split in ["train", "val", "test"]:
    img_dir = DATASET_PATH / "images" / split
    lbl_dir = DATASET_PATH / "labels" / split
    if img_dir.exists():
        for img_path in sorted(img_dir.glob("*.*"))[:10]:
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                sample_images.append((img_path, lbl_path))

# Display grid
n = min(len(sample_images), 20)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for i in range(n):
    img_path, lbl_path = sample_images[i]
    img = draw_annotations(img_path, lbl_path, CLASS_NAMES)
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(img_path.stem)

for i in range(n, len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance analysis
print("=" * 60)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 60)

total_anns = sum(stats["annotations_per_class"].values())
if total_anns > 0:
    for cls_id in range(len(CLASS_NAMES)):
        count = stats["annotations_per_class"].get(cls_id, 0)
        pct = (count / total_anns) * 100
        bar = "█" * int(pct / 2)
        print(f"{CLASS_NAMES[cls_id]:<20} {count:>6} ({pct:>5.1f}%) {bar}")

    # Check for severe imbalance
    max_count = max(stats["annotations_per_class"].values())
    min_count = min(stats["annotations_per_class"].values())
    if min_count > 0:
        ratio = max_count / min_count
        print(f"\nMax/Min ratio: {ratio:.1f}")
        if ratio > 10:
            print("⚠️  SEVERE class imbalance detected!")
            print("Consider:")
            print("  - Collecting more minority examples")
            print("  - Using class weights")
            print("  - Careful augmentation for minority classes")
        elif ratio > 5:
            print("⚠️  Moderate class imbalance detected.")
        else:
            print("✅ Class distribution is reasonably balanced.")

In [ ]:
# Final validation summary
print("=" * 60)
print("DATASET VALIDATION SUMMARY")
print("=" * 60)

issues = []
if stats["missing_labels"]:
    issues.append(f"Missing labels: {len(stats['missing_labels'])}")
if stats["empty_labels"]:
    issues.append(f"Empty labels: {len(stats['empty_labels'])}")
if stats["invalid_coords"]:
    issues.append(f"Invalid coordinates: {len(stats['invalid_coords'])}")

if issues:
    print("❌ Issues found:")
    for issue in issues:
        print(f"  - {issue}")
    print("\nPlease fix these issues before training.")
else:
    print("✅ Dataset validation passed!")
    print("Ready to proceed to training.")